# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

I have set the RDD patent into its own function so this would be the parameters used to simulate a real world implementation.

In [ ]:
citing_name='"CITING"'
cited_name='"CITED"'
patent_name='"PATENT"'
co_state='"POSTATE"'
takes: int | None = None

This brings the header that will be used for the joining.

In [ ]:
citations_header = rddCitations.first().split(",")
patents_header = rddPatents.first().split(",")

I am grabbing the indices of the names that I have defined/passed above.

In [ ]:
citing_idx = citations_header.index(citing_name)
cited_idx = citations_header.index(cited_name)
patent_idx = patents_header.index(patent_name)
state_idx = patents_header.index(co_state)

I am only getting the citing column for the joining process.

In [ ]:
citations_clean = rddCitations.filter(
  lambda l: not l.startswith(citing_name)
)

I am only getting the patent column for the joining process.

In [ ]:
patents_clean = rddPatents.filter(lambda l: not l.startswith(patent_name))

To be able to do the join of the cited based on the citings, this will split the elements and switch the elements around to do the join.

In [ ]:
citations_by_cited = citations_clean.map(lambda l: l.split(",")).map(
    lambda p: (p[cited_idx], p[citing_idx])
)

This makes sure that the labels don't have the comma and the patent and state columns don't have the extra quotations.

In [ ]:
patents_state = patents_clean.map(lambda l: l.split(",")).map(
        lambda p: (p[patent_idx], p[state_idx].replace('"', ""))
    )

Does the cited join on the patents then maps back to the citing and cited columns to perform the citing join on the patent table.

In [ ]:
cited_joined = citations_by_cited.join(patents_state)

citing_keyed = cited_joined.map(lambda x: (x[1][0], x[1][1]))

citing_joined = citing_keyed.join(patents_state)

This filters the null and empty elements in the state columns while checking if they are the same state/citing to patent.

In [ ]:
filtered_matches = citing_joined.filter(
    lambda x: x[1][0] != "" and x[1][1] != "" and x[1][0] == x[1][1]
)

This counts the same state that was filtered. It first maps all of the first elements to one then does the count.

In [ ]:
same_state_counts = filtered_matches.map(lambda x: (x[0], 1)).reduceByKey(
    lambda a, b: a + b
)

This maps the patents to the split of the elements. This mapping is then left joined by the counts made.

In [ ]:
patents_by_id = patents_clean.map(lambda l: (l.split(",")[patent_idx], l))
result = patents_by_id.leftOuterJoin(same_state_counts)

In this chunk of code, I added a new column name for the count of same cited/citing patents to the original patent table. Then made sure that it is referring to the original RDD instead of creating a new RDD. Then it gets sorted by match count in descending order and append match count to the raw CSV. Then I added back the header to the new RDD since I have not used the header previously. The if statement is there if there is a value for the take function else it just shows the full RDD.

In [ ]:
header_string = f'{rddPatents.first()}, "CO_STATE"'
header_rdd = sc.parallelize([header_string])

final_rdd = result.map(lambda x: (x[1][0], x[1][1] if x[1][1] is not None else 0))\
  .sortBy(lambda x: x[1], ascending=False)\
  .map(lambda x: f"{x[0]},{x[1]}")

header_rdd.union(final_rdd)
if takes is not None:
    header_rdd.take(takes)
else:
    header_rdd

Here is the full function.

In [ ]:
from pyspark import SparkConf, SparkContext


def patent_RDD(
    citing_name='"CITING"',
    cited_name='"CITED"',
    patent_name='"PATENT"',
    co_state='"POSTATE"',
    takes: int | None = None,
):
    conf = SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
    sc = SparkContext.getOrCreate(conf=conf)

    rddCitations = sc.textFile("cite75_99.txt.gz")
    rddPatents = sc.textFile("apat63_99.txt.gz")

    citations_header = rddCitations.first().split(",")
    patents_header = rddPatents.first().split(",")

    citing_idx = citations_header.index(citing_name)
    cited_idx = citations_header.index(cited_name)
    patent_idx = patents_header.index(patent_name)
    state_idx = patents_header.index(co_state)

    citations_clean = rddCitations.filter(lambda l: not l.startswith(citing_name))
    patents_clean = rddPatents.filter(lambda l: not l.startswith(patent_name))

    citations_by_cited = citations_clean.map(lambda l: l.split(",")).map(
        lambda p: (p[cited_idx], p[citing_idx])
    )
    patents_state = patents_clean.map(lambda l: l.split(",")).map(
        lambda p: (p[patent_idx], p[state_idx].replace('"', ""))
    )

    cited_joined = citations_by_cited.join(patents_state)

    citing_keyed = cited_joined.map(lambda x: (x[1][0], x[1][1]))

    citing_joined = citing_keyed.join(patents_state)

    filtered_matches = citing_joined.filter(
        lambda x: x[1][0] != "" and x[1][1] != "" and x[1][0] == x[1][1]
    )

    same_state_counts = filtered_matches.map(lambda x: (x[0], 1)).reduceByKey(
        lambda a, b: a + b
    )

    patents_by_id = patents_clean.map(lambda l: (l.split(",")[patent_idx], l))
    result = patents_by_id.leftOuterJoin(same_state_counts)

    header_string = f'{rddPatents.first()}, "CO_STATE"'
    header_rdd = sc.parallelize([header_string])

    final_rdd = result.map(lambda x: (x[1][0], x[1][1] if x[1][1] is not None else 0))\
      .sortBy(lambda x: x[1], ascending=False)\
      .map(lambda x: f"{x[0]},{x[1]}")

    header_rdd.union(final_rdd)
    if takes is not None:
        return header_rdd.take(takes)
    return header_rdd

To perform the function run this main function/file below.

In [ ]:
from dataframe import patent_DataFrame
from rdd import patent_RDD
def main(df_rdd:str):
    if df_rdd is "dataframe":
        df = patent_DataFrame()
        df.show()
    elif df_rdd.upper() is "RDD":
        results = patent_RDD(takes=5)
        print(results)
    else:
        raise ValueError("Options are dataframe or RDD")

if __name__ == "__main__":
    main("rdd")